Load the function from recommendation and run this notebook to talk to your recommendation agent

In [0]:
%run "./04_recommendation"

In [0]:
import requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
WORKSPACE_URL = ctx.apiUrl().getOrElse(None)
TOKEN = ctx.apiToken().getOrElse(None)

def call_chat_model(messages, max_tokens=200):
    url = f"{WORKSPACE_URL}/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations"
    headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json"
    }
    payload = {"messages": messages, "max_tokens": max_tokens}
    resp = requests.post(url, headers=headers, json=payload)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

# test
call_chat_model([{"role": "user", "content": "Say hello in one word."}])

In [0]:
def parse_intent(user_question):
    system_prompt = """You extract structured info from a movie recommendation question.
Return ONLY valid JSON, no other text, in this exact format:
{"genre": "<one of: Action, Adventure, Animation, Comedy, Crime, Documentary, Drama, Family, Fantasy, History, Horror, Music, Mystery, Romance, Science Fiction, TV Movie, Thriller, War, Western, or null>", "country": "<IN or IE or null>", "movie_reference": "<movie title or null>", "top_n": <number, default 5>, "query_text": "<a short descriptive phrase capturing the mood/theme of the request>", "sort_by": "<one of: popularity, rating, similarity>", "language": "<full language name like English, Hindi, French, or 'any' if user says any/all languages, default English if not mentioned>"}

Use "popularity" for sort_by if the user says "popular", "trending", "well-known", or similar.
Use "rating" if the user says "best", "highest rated", "top rated".
Use "similarity" as default if the user just describes a mood/theme with no explicit ranking preference.

If the request implies multiple genres (e.g. "romcom" = Romance + Comedy), pick the single most dominant genre only.
Country must be "IN" for India or "IE" for Ireland. If no country mentioned, use null.
If the user references a specific movie they liked, put it in movie_reference.
If no language is mentioned, default language to "English".
"""
    raw = call_chat_model([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ], max_tokens=200)
    
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)



In [0]:
def format_response(recs, country):
    if recs.empty:
        return "I couldn't find any matching movies. Try a different genre or description."
    
    lines = [f"Here are my picks (streaming in {country}):\n"]
    for _, row in recs.iterrows():
        platforms = ", ".join(row["available_on"]) if row.get("available_on") else "Not available to stream"
        lines.append(f"🎬 **{row['title']}** — {platforms}")
    return "\n".join(lines)


def chatbot(user_question):
    intent = parse_intent(user_question)
    
    country = intent.get("country") or "IN"
    top_n = intent.get("top_n") or 5
    genre = intent.get("genre")
    movie_ref = intent.get("movie_reference")
    query_text = intent.get("query_text") or user_question
    sort_by = intent.get("sort_by") or "similarity"
    
    if movie_ref:
        match = embeddings_pdf[embeddings_pdf["title"].str.lower() == movie_ref.lower()]
        if match.empty:
            return f"I couldn't find '{movie_ref}' in my catalog. Try another title?"
        recs = recommend_similar_to_movie(movie_ref, top_n=top_n)
        recs = attach_watch_info(recs, country)
    
    elif genre:
        recs = recommend_by_genre_filtered(genre, query_text, country=country, top_n=top_n, sort_by=sort_by)
    
    else:
        recs = recommend_by_text(query_text, top_n=top_n)
        recs = attach_watch_info(recs, country)
    
    return format_response(recs, country)

In [0]:
dbutils.widgets.text("user_question", "")
question = dbutils.widgets.get("user_question")

if question:
    print(chatbot(question))